# Estimate Mode Choice

Estimate mode choice models of TNC vs transit/walk

In [17]:
import numpy as np

import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.reset_option('display.float_format')

import biogeme.biogeme as bio
import biogeme.database as biodb
from biogeme import models
from biogeme.expressions import Beta, Variable

In [18]:
# read the data
df = pd.read_csv('out/combined_estimation_file.csv')
df.head()

C:\Users\ger225\AppData\Local\Temp\ipykernel_29824\360866534.py:2: DtypeWarning: Columns (0: depart_date, 1: linked_trip_mode_labeled, 2: income_labeled) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('out/combined_estimation_file.csv')


,Unnamed: 0,hh_id,person_id,person_num,day_id,day_num,depart_date,o_tract_2020,d_tract_2020,linked_trip_id,linked_trip_num,linked_trip_mode,linked_trip_weight,linked_trip_mode_labeled,mode,mode2,distance_miles,duration_minutes,o_district,d_district,o_community,d_community,time_period,ff_car_time_minutes,car_ivt,tnc_wait,tnc_time,tnc_fare,transit_fare,walk_time,transit_time,transit_or_walk_time,walk_faster_than_transit,transit_or_walk_fare,income_broad,income_labeled,hh_share_inc_under_100k,hh_share_inc_over_100k,tnc_trip_id,obs_fare,obs_tip,obs_additional_charges,transit_avail,walk_avail,tnc_time_minus_transit_walk,tnc_cost_minus_transit_walk,transit_or_walk_avail
0,0,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031320101,17031081500,2.400012e+15,1.0,15.0,1853.792592,Walk,walk,walk,0.810270,20.0,Downtown,Downtown,32.0,8.0,midday,3.738333,6.186942,5,11.186942,6.707731,2.5,16.205401,22.0,16.205401,True,0.0,5.0,"$150,000 or more",0.341000,0.659000,NaN,NaN,NaN,NaN,1,1,-5.018459,6.707731,1
1,1,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031081500,17031081403,2.400012e+15,2.0,15.0,1853.792592,Walk,walk,walk,0.338027,28.0,Downtown,Downtown,8.0,8.0,midday,1.660000,2.747300,5,7.747300,4.798943,2.5,6.760535,7.0,6.760535,True,0.0,5.0,"$150,000 or more",0.321678,0.678322,NaN,NaN,NaN,NaN,1,1,0.986765,4.798943,1
2,2,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031081403,17031320101,2.400012e+15,3.0,15.0,1853.792592,Walk,walk,walk,0.549293,15.0,Downtown,Downtown,8.0,32.0,midday,3.421667,5.662858,5,10.662858,6.244886,2.5,10.985870,23.0,10.985870,True,0.0,5.0,"$150,000 or more",0.460539,0.539461,NaN,NaN,NaN,NaN,1,1,-0.323012,6.244886,1
3,3,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031320101,17031320101,2.400012e+15,4.0,15.0,1853.792592,Walk,walk,walk,0.319386,16.0,Downtown,Downtown,32.0,32.0,midday,2.201667,3.643758,5,8.643758,5.167457,2.5,6.387712,12.0,6.387712,True,0.0,5.0,"$150,000 or more",0.341000,0.659000,NaN,NaN,NaN,NaN,1,1,2.256047,5.167457,1
4,4,24000124.0,2.400012e+09,2.0,2.400012e+11,1.0,2024-05-21,17031320101,17031320102,2.400012e+15,1.0,15.0,1853.792592,Walk,walk,walk,0.751861,16.0,Downtown,Downtown,32.0,32.0,midday,2.201667,3.643758,5,8.643758,5.561010,2.5,15.037220,12.0,12.000000,False,2.5,5.0,"$150,000 or more",0.341000,0.659000,NaN,NaN,NaN,NaN,1,1,-3.356242,3.061010,1


In [19]:
# which columns have NaNs, and how many
df.isna().sum()


Unnamed: 0                          0
hh_id                          156346
person_id                      156346
person_num                     156346
day_id                         156346
day_num                        156346
depart_date                    156346
o_tract_2020                        0
d_tract_2020                        0
linked_trip_id                 156346
linked_trip_num                156346
linked_trip_mode               156346
linked_trip_weight                  0
linked_trip_mode_labeled       156346
mode                                0
mode2                               0
distance_miles                      0
duration_minutes                    0
o_district                          0
d_district                          0
o_community                         0
d_community                         0
time_period                         0
ff_car_time_minutes                 0
car_ivt                             0
tnc_wait                            0
tnc_time    

In [20]:
# drop recrods where we don't know the income shares

trips_before = len(df)

df = df[df['hh_share_inc_under_100k']>0]
df = df[df['hh_share_inc_under_100k']>0]

print("Before: " + str(trips_before) + " After: " + str(len(df)))

Before: 161445 After: 161301


In [21]:
# fill the remaining missing values with zeros to make biogeme happy--BE CAREFUL!
df = df.fillna(0)

In [22]:
# calculate normalized weights
df['normalized_weights'] = df['linked_trip_weight'] / df['linked_trip_weight'].sum() * len(df)

In [23]:
# add a flag for trips made by people in HHs with <$100k, $100k+ and missing annual income
# income_broad: 
# 1	Under $30,000
# 2	$30,000-$59,999
# 3	$60,000-$99,999
# 4	$100,000-$149,999
# 5	$150,000 or more
# 999	Prefer not to answer

df['inc_under_100k'] = np.where(df['income_broad']<=3, 1, 0)
df['inc_over_100k']  = np.where((df['income_broad']==4) | (df['income_broad']==5), 1, 0)
df['inc_missing']    = np.where((df['income_broad']==999), 1, 0)

In [24]:
# Biogeme needs a NUMERIC choice column: tnc=1, transit=2, walk=3 and only numeric values in its database format. 
df['CHOICE'] = df['mode'].map({'tnc': 1, 'transit': 2, 'walk': 3})
df['BINARY_CHOICE'] = df['mode'].map({'tnc': 1, 'transit': 2, 'walk': 2})

df_numeric = df.select_dtypes(include='number').copy()

db = biodb.Database('mode_choice', df_numeric)

# Initial Specification

In [25]:
# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)   
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST      = Beta('B_COST',      0, None, None, 0)   # generic, shared across modes

# --- variables ---
tnc_time     = Variable('tnc_time')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare')
transit_fare = Variable('transit_fare')
CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST * tnc_fare
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST * transit_fare
V_walk    = ASC_WALK    + B_TIME * walk_time

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, logprob)
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST']['Value']
print("\nValue of Time: " + str(round(vot, 2)))


Results for model mnl_mode_choice
Nbr of parameters:		4
Sample size:			161301
Excluded data:			0
Null log likelihood:		-139436.4
Final log likelihood:		-19260.97
Likelihood ratio test (null):		240350.9
Rho square (null):			0.862
Rho bar square (null):			0.862
Akaike Information Criterion:	38529.95
Bayesian Information Criterion:	38569.91

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT -2.838428      0.032206   -88.133440           0.0
ASC_WALK    -1.698611      0.029626   -57.335793           0.0
B_COST      -0.072387      0.002928   -24.719191           0.0
B_TIME      -0.148114      0.003058   -48.435323           0.0

Value of Time: 122.77


# Try weighted estimation

Normally I would use an unweighted estimation.  Here I try a weighted estimation since I think it will affect primarily the ASCs.  What we see below is that it is that the time and cost coefficients are smaller in magnitude, but the ASCs aren't much different.  I think we're better off sticking with the unweighted estimation, which is the norm. 

In [26]:
# Normally I would use an unweighted estimation, but here I care about the ASCs, so I will try weighting it.

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)   
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST      = Beta('B_COST',      0, None, None, 0)   # generic, shared across modes

# --- variables ---
tnc_time     = Variable('tnc_time')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare')
transit_fare = Variable('transit_fare')
CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST * tnc_fare
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST * transit_fare
V_walk    = ASC_WALK    + B_TIME * walk_time

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST']['Value']
print("\nValue of Time: " + str(round(vot, 2)))

Results for model mnl_mode_choice
Nbr of parameters:		4
Sample size:			161301
Excluded data:			0
Null log likelihood:		-139436.4
Final log likelihood:		-85342.49
Likelihood ratio test (null):		108187.9
Rho square (null):			0.388
Rho bar square (null):			0.388
Akaike Information Criterion:	170693
Bayesian Information Criterion:	170732.9

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT  2.453656      0.026181    93.719919           0.0
ASC_WALK     3.432971      0.029684   115.649287           0.0
B_COST      -0.063512      0.002529   -25.114021           0.0
B_TIME      -0.053592      0.001118   -47.921005           0.0

Value of Time: 50.63


# Consider zonal incomes instead of HH level incomes

In the TNC data, we won't actually observe the HH level incomes due to privacy restrictions.  Instead try segmenting VOT based on the income distribution in the Census tract of the trip's origin. 

Again, we observe it is kind of backwards, which is strange. Maybe I should not have dropped the short walk trips? 

In [27]:
# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)   
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k  = Variable('hh_share_inc_over_100k')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k 
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST_LOW * transit_fare * hh_share_inc_under_100k + B_COST_HI * transit_fare * hh_share_inc_over_100k 
V_walk    = ASC_WALK    + B_TIME * walk_time

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, logprob)
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))



Results for model mnl_mode_choice
Nbr of parameters:		5
Sample size:			161301
Excluded data:			0
Null log likelihood:		-139436.4
Final log likelihood:		-19252.78
Likelihood ratio test (null):		240367.3
Rho square (null):			0.862
Rho bar square (null):			0.862
Akaike Information Criterion:	38515.56
Bayesian Information Criterion:	38565.52

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT -2.844246      0.032313   -88.020844           0.0
ASC_WALK    -1.710203      0.029923   -57.153974           0.0
B_COST_HI   -0.089486      0.005180   -17.274090           0.0
B_COST_LOW  -0.058896      0.004185   -14.074078           0.0
B_TIME      -0.148492      0.003081   -48.198105           0.0

Value of Time for HH <$100k: 151.28
Value of Time for HH $100k+: 99.56


In [28]:
# weighted estimation with zonal incomes
# Normally I would use an unweighted estimation, but here I care about the ASCs, so I will try weighting it.

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)   
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare')
transit_fare = Variable('transit_fare')
CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k 
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST_LOW * transit_fare * hh_share_inc_under_100k + B_COST_HI * transit_fare * hh_share_inc_over_100k 
V_walk    = ASC_WALK    + B_TIME * walk_time

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))


Results for model mnl_mode_choice
Nbr of parameters:		5
Sample size:			161301
Excluded data:			0
Null log likelihood:		-139436.4
Final log likelihood:		-85323.99
Likelihood ratio test (null):		108224.9
Rho square (null):			0.388
Rho bar square (null):			0.388
Akaike Information Criterion:	170658
Bayesian Information Criterion:	170707.9

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT  2.469000      0.026136    94.467753           0.0
ASC_WALK     3.455864      0.029601   116.749774           0.0
B_COST_HI   -0.046958      0.003875   -12.116939           0.0
B_COST_LOW  -0.073482      0.003318   -22.146742           0.0
B_TIME      -0.053676      0.001127   -47.634983           0.0

Value of Time for HH <$100k: 43.83
Value of Time for HH $100k+: 68.58


That's getting closer.  The ACSs seem high, as are the VOTs, but they are at least in the right order. 

In [29]:
# weighted estimation with zonal incomes
# Exclue the transit fare, which is probably discounted for most transit riders. 

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)   
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare')
transit_fare = Variable('transit_fare')
CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k 
V_transit = ASC_TRANSIT + B_TIME * transit_time 
V_walk    = ASC_WALK    + B_TIME * walk_time

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))


Results for model mnl_mode_choice
Nbr of parameters:		5
Sample size:			161301
Excluded data:			0
Null log likelihood:		-139436.4
Final log likelihood:		-85275.7
Likelihood ratio test (null):		108321.5
Rho square (null):			0.388
Rho bar square (null):			0.388
Akaike Information Criterion:	170561.4
Bayesian Information Criterion:	170611.3

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT  2.335439      0.030192    77.354124           0.0
ASC_WALK     3.473101      0.029429   118.016987           0.0
B_COST_HI   -0.034830      0.003544    -9.828127           0.0
B_COST_LOW  -0.080848      0.003346   -24.162788           0.0
B_TIME      -0.053859      0.001137   -47.351846           0.0

Value of Time for HH <$100k: 39.97
Value of Time for HH $100k+: 92.78


That's a better VOT distribution, although still very high. 

In [30]:
# unweighted.  Assume transit is free. 

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)   
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k  = Variable('hh_share_inc_over_100k')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k 
V_transit = ASC_TRANSIT + B_TIME * transit_time 
V_walk    = ASC_WALK    + B_TIME * walk_time

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, logprob)
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))



Results for model mnl_mode_choice
Nbr of parameters:		5
Sample size:			161301
Excluded data:			0
Null log likelihood:		-139436.4
Final log likelihood:		-19251.95
Likelihood ratio test (null):		240368.9
Rho square (null):			0.862
Rho bar square (null):			0.862
Akaike Information Criterion:	38513.9
Bayesian Information Criterion:	38563.86

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT -3.028796      0.035458   -85.418863           0.0
ASC_WALK    -1.710066      0.029908   -57.177407           0.0
B_COST_HI   -0.088784      0.004946   -17.950490           0.0
B_COST_LOW  -0.059577      0.004014   -14.843467           0.0
B_TIME      -0.148480      0.003081   -48.199463           0.0

Value of Time for HH <$100k: 149.53
Value of Time for HH $100k+: 100.34


Still flipped, and even higher VOTs. 

What if I add a constant segmented by income? 

In [31]:
# weighted estimation with zonal incomes
# Exclue the transit fare, which is probably discounted for most transit riders. 
# Add constant segmented by income

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)   
ASC_TRANSIT_HI = Beta('ASC_TRANSIT_HI', 0, None, None, 0) # constant on share of high-income HHs
ASC_WALK_HI    = Beta('ASC_WALK_HI',    0, None, None, 0) # constant on share of high-income HHs  
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare')
transit_fare = Variable('transit_fare')
CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k 
V_transit = ASC_TRANSIT + B_TIME * transit_time + ASC_TRANSIT_HI * hh_share_inc_over_100k  
V_walk    = ASC_WALK    + B_TIME * walk_time    + ASC_WALK_HI * hh_share_inc_over_100k  

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))


Results for model mnl_mode_choice
Nbr of parameters:		7
Sample size:			161301
Excluded data:			0
Null log likelihood:		-139436.4
Final log likelihood:		-85109.72
Likelihood ratio test (null):		108653.4
Rho square (null):			0.39
Rho bar square (null):			0.39
Akaike Information Criterion:	170233.4
Bayesian Information Criterion:	170303.4

                   Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT     2.582867      0.092218    28.008119  0.000000e+00
ASC_TRANSIT_HI -0.523720      0.175872    -2.977845  2.902824e-03
ASC_WALK        3.373216      0.090024    37.470258  0.000000e+00
ASC_WALK_HI     0.189338      0.171385     1.104753  2.692668e-01
B_COST_HI      -0.048052      0.005885    -8.164724  2.220446e-16
B_COST_LOW     -0.070164      0.005799   -12.100103  0.000000e+00
B_TIME         -0.054270      0.001155   -46.972603  0.000000e+00

Value of Time for HH <$100k: 46.41
Value of Time for HH $100k+: 67.76


In [32]:
# unweighted estimation with zonal incomes
# Exclue the transit fare, which is probably discounted for most transit riders. 
# Add constant segmented by income

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)   
ASC_TRANSIT_HI = Beta('ASC_TRANSIT_HI', 0, None, None, 0) # constant on share of high-income HHs
ASC_WALK_HI    = Beta('ASC_WALK_HI',    0, None, None, 0) # constant on share of high-income HHs  
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare')
transit_fare = Variable('transit_fare')
CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k 
V_transit = ASC_TRANSIT + B_TIME * transit_time + ASC_TRANSIT_HI * hh_share_inc_over_100k  
V_walk    = ASC_WALK    + B_TIME * walk_time    + ASC_WALK_HI * hh_share_inc_over_100k  

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))


Results for model mnl_mode_choice
Nbr of parameters:		7
Sample size:			161301
Excluded data:			0
Null log likelihood:		-139436.4
Final log likelihood:		-19250.37
Likelihood ratio test (null):		240372.1
Rho square (null):			0.862
Rho bar square (null):			0.862
Akaike Information Criterion:	38514.74
Bayesian Information Criterion:	38584.68

                   Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT    -3.203267      0.087907   -36.439222      0.000000
ASC_TRANSIT_HI  0.356485      0.165189     2.158040      0.030925
ASC_WALK       -1.746386      0.059925   -29.142952      0.000000
ASC_WALK_HI     0.079684      0.113011     0.705101      0.480747
B_COST_HI      -0.080156      0.007274   -11.019435      0.000000
B_COST_LOW     -0.067330      0.005704   -11.804014      0.000000
B_TIME         -0.148435      0.003082   -48.159449      0.000000

Value of Time for HH <$100k: 132.28
Value of Time for HH $100k+: 111.11


What if I use the observed TNC time and fare when they are available? 

In [41]:
# use the observed TNC time and fare where they are avialable

df['tnc_time_2'] = np.where(df['mode']=='tnc', 5+df['duration_minutes'], 5+df['car_ivt'])
df['tnc_fare_2'] = np.where(df['mode']=='tnc', df['obs_fare'], df['tnc_fare'])

df_numeric = df.select_dtypes(include='number').copy()

db = biodb.Database('mode_choice', df_numeric)


In [42]:
# weighted estimation with zonal incomes
# Exclue the transit fare, which is probably discounted for most transit riders. 
# Add constant segmented by income
# use observed TNC time/fare where available

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)   
ASC_TRANSIT_HI = Beta('ASC_TRANSIT_HI', 0, None, None, 0) # constant on share of high-income HHs
ASC_WALK_HI    = Beta('ASC_WALK_HI',    0, None, None, 0) # constant on share of high-income HHs  
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time_2')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare_2')
transit_fare = Variable('transit_fare')
CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k 
V_transit = ASC_TRANSIT + B_TIME * transit_time + ASC_TRANSIT_HI * hh_share_inc_over_100k  
V_walk    = ASC_WALK    + B_TIME * walk_time    + ASC_WALK_HI * hh_share_inc_over_100k  

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))


Results for model mnl_mode_choice
Nbr of parameters:		7
Sample size:			161301
Excluded data:			0
Null log likelihood:		-139436.4
Final log likelihood:		-85477.01
Likelihood ratio test (null):		107918.8
Rho square (null):			0.387
Rho bar square (null):			0.387
Akaike Information Criterion:	170968
Bayesian Information Criterion:	171038

                   Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT     1.422980      0.086049    16.536772      0.000000
ASC_TRANSIT_HI  1.468759      0.167819     8.752053      0.000000
ASC_WALK        2.289428      0.082696    27.684890      0.000000
ASC_WALK_HI     2.055945      0.161647    12.718763      0.000000
B_COST_HI       0.016733      0.006087     2.748794      0.005982
B_COST_LOW     -0.149105      0.006506   -22.917356      0.000000
B_TIME         -0.051298      0.001092   -46.970553      0.000000

Value of Time for HH <$100k: 20.64
Value of Time for HH $100k+: -183.94


In [43]:
# weighted estimation with zonal incomes
# Exclue the transit fare, which is probably discounted for most transit riders. 
# Add constant segmented by income
# use observed TNC time/fare where available

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)    
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time_2')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare_2')
transit_fare = Variable('transit_fare')
CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k 
V_transit = ASC_TRANSIT + B_TIME * transit_time  
V_walk    = ASC_WALK    + B_TIME * walk_time    

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))


Results for model mnl_mode_choice
Nbr of parameters:		5
Sample size:			161301
Excluded data:			0
Null log likelihood:		-139436.4
Final log likelihood:		-85693.43
Likelihood ratio test (null):		107486
Rho square (null):			0.385
Rho bar square (null):			0.385
Akaike Information Criterion:	171396.9
Bayesian Information Criterion:	171446.8

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT  2.131185      0.030171    70.636754  0.000000e+00
ASC_WALK     3.285053      0.029145   112.712229  0.000000e+00
B_COST_HI   -0.031280      0.003936    -7.946492  1.998401e-15
B_COST_LOW  -0.102936      0.003549   -29.006201  0.000000e+00
B_TIME      -0.050673      0.001081   -46.854952  0.000000e+00

Value of Time for HH <$100k: 29.54
Value of Time for HH $100k+: 97.2


We're getting closer.  I think I can be smarter about using more realistic times and costs for all options.  